# 3. RAG Pipeline with LangChain on Amazon Bedrock

## Overview

Retrieval-Augmented Generation solves the problem that a foundation model knows nothing about
**your** data. Instead of fine-tuning, we retrieve the relevant passages at query time and put
them into the prompt.

In this notebook you will build the full pipeline over a real document — the official
**"Medicare & You" handbook** published by the Centers for Medicare & Medicaid Services — using
**Amazon Titan Text Embeddings V2** on Bedrock and a local **Chroma** vector store.

The handbook is a good stand-in for the documents a payer or provider actually indexes: an
evidence of coverage, a summary of benefits, a provider manual, a policy handbook. It is long,
it is written for members rather than engineers, it changes every year, and — critically — it is
page-numbered, so every answer can carry a citation a human can go and check.

Pipeline:

```
  INDEXING (once)                       RETRIEVAL + GENERATION (per query)
  ---------------                       ---------------------------------
  Load PDF                              Member question
     |                                        |
  Split into chunks                     Embed question
     |                                        |
  Embed chunks (Titan V2)               Similarity search in Chroma
     |                                        |
  Store in Chroma  <--------------------  Top-k chunks
                                              |
                                        Prompt = question + chunks
                                              |
                                        Claude on Bedrock -> grounded answer
```


## Concepts

- **Document loaders** turn a source into `Document` objects (`page_content` + `metadata`)
- **Text splitters** cut documents into retrievable chunks
- **Embeddings** map text to vectors so that similar meaning is geometrically close
- **Vector stores** index those vectors for fast similarity search
- **Retrievers** are the Runnable interface over a vector store
- **Generation** puts the retrieved chunks into the prompt and asks the model to answer

> **Why citations are the point of this notebook.** In a benefits assistant, an unsourced answer
> is close to useless: a member services representative cannot act on it, a compliance reviewer
> cannot approve it, and a member cannot verify it. The page citation is not decoration. Build
> the habit here — every generation cell in this notebook cites, and section 5.1 tests what
> happens when the corpus does not contain the answer.
>
> **Sample data, current document.** The handbook downloaded below is the official current
> edition. It is a US Government work in the public domain. It is still a general reference:
> a specific member's plan may differ, and nothing here should drive a real coverage decision.

#### 0. Setup

This notebook needs `medicare-and-you.pdf` in a `data/` folder next to the notebook. The next
cell downloads it into your SageMaker space if it is not already there, so there is nothing to
copy across from your laptop.

The Chroma index and the PDF are written to your JupyterLab **home directory**, which lives on
the EBS volume attached to the space. Both survive a space restart.

In [1]:
# ---------------------------------------------------------------------------
# SageMaker Studio setup
#
# There is no .env file and no access key in this lab. boto3 automatically
# picks up short-lived credentials from the SageMaker execution role attached
# to this space, which is both simpler and safer than a long-lived key pair.
# ---------------------------------------------------------------------------
import os
import boto3

# Leave this as None to use the region this SageMaker space is running in.
# Set it ONLY if you enabled Bedrock model access in a different region.
BEDROCK_REGION = None

AWS_REGION = BEDROCK_REGION or boto3.session.Session().region_name or "us-east-1"
CHAT_MODEL_ID = os.environ.get("BEDROCK_CHAT_MODEL_ID", "amazon.nova-pro-v1:0")
EMBED_MODEL_ID = os.environ.get("BEDROCK_EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")

from langchain_aws import ChatBedrockConverse, BedrockEmbeddings

session = boto3.Session(region_name=AWS_REGION)

# Show which IAM identity this notebook is using. On SageMaker this is the
# execution role of your space - the role that needs Bedrock permissions.
identity = session.client("sts").get_caller_identity()
EXECUTION_ROLE_ARN = identity["Arn"]

runtime = session.client("bedrock-runtime")

model = ChatBedrockConverse(model=CHAT_MODEL_ID, client=runtime, temperature=0, max_tokens=1024)

print("Region         :", AWS_REGION)
print("Execution role :", EXECUTION_ROLE_ARN)
print("Chat model ready:", CHAT_MODEL_ID)

In [2]:
# Fetch the Medicare & You handbook into ./data if it is not already present.
# This is a US Government work published by CMS and is in the public domain.
#
# SageMaker spaces have internet access by default. If yours is in a VPC with no
# egress, download the PDF on your own machine from
# https://www.medicare.gov/publications/10050-medicare-and-you.pdf
# and upload it via the JupyterLab file browser to ./data/medicare-and-you.pdf
import os
import urllib.request

PDF_DIR = "./data"
PDF_PATH = os.path.join(PDF_DIR, "medicare-and-you.pdf")
PDF_URL = "https://www.medicare.gov/publications/10050-medicare-and-you.pdf"

os.makedirs(PDF_DIR, exist_ok=True)

if os.path.exists(PDF_PATH) and os.path.getsize(PDF_PATH) > 1_000_000:
    print("Already present:", PDF_PATH, f"({os.path.getsize(PDF_PATH) / 1e6:.1f} MB)")
else:
    try:
        print("Downloading the Medicare & You handbook ...")
        request = urllib.request.Request(
            PDF_URL, headers={"User-Agent": "Mozilla/5.0 (agentic-ai-lab)"}
        )
        with urllib.request.urlopen(request, timeout=120) as resp, open(PDF_PATH, "wb") as out:
            out.write(resp.read())
        print("Saved to", PDF_PATH, f"({os.path.getsize(PDF_PATH) / 1e6:.1f} MB)")
    except Exception as err:
        print("Download failed:", err)
        print()
        print("Download it manually from:")
        print("   ", PDF_URL)
        print("then upload it to the data/ folder as medicare-and-you.pdf and re-run this cell.")

---
## 1. Documents and document loaders

A `Document` is LangChain's unit of text. It has `page_content` (the string), `metadata`
(a dict describing where it came from) and an optional `id`.

#### 1.1. A hand-made document

In [3]:
from langchain_core.documents import Document

sample_docs = [
    Document(
        page_content="Part B covers doctor visits, outpatient care and durable medical equipment.",
        metadata={"source": "plan-notes", "topic": "part_b"},
    ),
    Document(
        page_content="Part D is prescription drug coverage, offered through private plans.",
        metadata={"source": "plan-notes", "topic": "part_d"},
    ),
]

print(sample_docs[0].page_content)
print(sample_docs[0].metadata)

#### 1.2. Loading a real PDF

`PyPDFLoader` produces **one Document per page**, with the file name and page number captured
in the metadata. That metadata is what lets us cite sources later.

The handbook runs to a few hundred pages, so this takes a few seconds.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./data/medicare-and-you.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print("Pages loaded:", len(docs))

In [5]:
# Look at a page from the middle of the document rather than the cover
sample_page = docs[min(40, len(docs) - 1)]
print(sample_page.page_content[:400])
print("\nMETADATA:", sample_page.metadata)

> **A metadata detail that will bite you.** `PyPDFLoader` reports a **zero-based PDF page
> index**, which is not the same as the page number printed on the page. In a member-facing tool
> that gap matters: you tell someone "see page 62" and they turn to a different page. Either add
> the offset when you display it, or extract the printed page number during ingestion and store
> that alongside. We keep the raw index in this lab so you can see the difference; fixing it is a
> worthwhile exercise.

#### 1.3. Splitting

A whole page is too coarse. If a page contains one relevant sentence and nine irrelevant
paragraphs, the relevant meaning gets "washed out" in the embedding.

`RecursiveCharacterTextSplitter` splits on paragraph breaks first, then lines, then words —
so chunks tend to break at natural boundaries. The **overlap** stops a sentence from being
severed from the context that gives it meaning.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Set PAGE_LIMIT to an integer (e.g. 120) for a faster, cheaper run during a
# time-boxed session. Leave it as None to index the whole handbook.
PAGE_LIMIT = None

pages_to_index = docs if PAGE_LIMIT is None else docs[:PAGE_LIMIT]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # characters per chunk
    chunk_overlap=200,     # characters shared between neighbouring chunks
    add_start_index=True,  # record where the chunk began in the original page
)

all_splits = text_splitter.split_documents(pages_to_index)

print("Pages being indexed:", len(pages_to_index))
print("Chunks created     :", len(all_splits))
print("\nExample chunk metadata:", all_splits[50].metadata)
print("\nExample chunk text:\n", all_splits[50].page_content[:300])

---
## 2. Embeddings with Amazon Titan

`BedrockEmbeddings` wraps the Bedrock embedding models. **Titan Text Embeddings V2** returns a
1024-dimension vector by default and supports 512 and 256 if you want a smaller index.

In [7]:
embeddings = BedrockEmbeddings(
    model_id=EMBED_MODEL_ID,
    client=runtime,
)

vector_1 = embeddings.embed_query("How much does Part B cost each month?")
vector_2 = embeddings.embed_query("What is the monthly premium for medical insurance?")
vector_3 = embeddings.embed_query("What is the boiling point of water?")

print("Vector length:", len(vector_1))
print("First 5 values:", vector_1[:5])

#### 2.1. Why embeddings work

Two sentences that mean the same thing produce vectors that point in almost the same
direction, even with no words in common. Cosine similarity measures that.

This is the property that makes a benefits assistant usable at all. Members do not use the
words the handbook uses — they say "the yearly amount I pay before insurance kicks in", not
"deductible". Keyword search fails on that; embeddings do not.

In [8]:
import numpy as np


def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Part B cost  vs  monthly premium :", round(cosine(vector_1, vector_2), 4))
print("Part B cost  vs  boiling water   :", round(cosine(vector_1, vector_3), 4))

# Member language vs handbook language
member_words = embeddings.embed_query("the amount I pay each year before my plan starts paying")
handbook_words = embeddings.embed_query("annual deductible")
print("member phrasing vs 'annual deductible':", round(cosine(member_words, handbook_words), 4))

---
## 3. Vector store — Chroma

Chroma is a lightweight open-source vector database. It runs in-process and persists to a
local folder, which makes it ideal for a lab. In production on AWS you would swap it for
Amazon OpenSearch Serverless or pgvector on Amazon Aurora — the LangChain interface is the
same, so only this cell would change.

#### 3.1. Initialise the store

In [9]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="medicare_handbook",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

print("Documents already indexed:", vector_store._collection.count())

#### 3.2. Index the chunks

This calls Bedrock once per batch of chunks and stores the vectors. The full handbook takes a
few minutes. Run it **once** — re-running adds duplicates.

In [10]:
import time

if vector_store._collection.count() == 0:
    start = time.time()
    ids = vector_store.add_documents(documents=all_splits)
    print(f"Indexed {len(ids)} chunks in {time.time() - start:.0f} seconds")
else:
    print("Collection already populated - skipping indexing")
    print("To rebuild from scratch, delete the ./chroma_db folder and re-run.")

#### 3.3. Similarity search

In [11]:
results = vector_store.similarity_search(
    "What is the late enrolment penalty for Part B?", k=2
)

for i, doc in enumerate(results, 1):
    print(f"--- RESULT {i} (pdf page index {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300], "\n")

#### 3.4. Search with relevance scores

In [12]:
scored = vector_store.similarity_search_with_score("Does Medicare cover hearing aids?", k=3)

for doc, score in scored:
    print(f"score={score:.4f}  page={doc.metadata.get('page')}")
    print(doc.page_content[:180].replace("\n", " "), "\n")

Chroma returns a **distance**, so lower is more similar. Different vector stores use
different conventions — always check before you threshold on a score.

#### 3.5. Asynchronous search

In [13]:
async_results = await vector_store.asimilarity_search(
    "How do I file an appeal if my claim is denied?", k=1
)
print(async_results[0].page_content[:300])

#### 3.6. Search by a pre-computed vector

In [14]:
query_vector = embeddings.embed_query("What is the difference between Original Medicare and Medicare Advantage?")
by_vector = vector_store.similarity_search_by_vector(query_vector, k=1)
print(by_vector[0].page_content[:300])

---
## 4. Retrievers

A retriever is the Runnable wrapper around search, so it can be piped into a chain. Chroma
supports three search types.

#### 4.1. Similarity (default)

Returns the k nearest chunks. Simple and fast; can return near-duplicates.

In [15]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

batch_results = similarity_retriever.batch([
    "When can I sign up for Medicare?",
    "What does Part A cover?",
])

for query_docs in batch_results:
    print("-", query_docs[0].page_content[:120].replace("\n", " "))

#### 4.2. Maximum Marginal Relevance (MMR)

MMR balances **relevance to the query** against **diversity among the results**. Use it when
your corpus repeats itself and plain similarity keeps returning the same paragraph three
times.

Benefits documents repeat themselves constantly — the same enrolment rule is restated in the
enrolment chapter, the costs chapter and the glossary. This is exactly the corpus MMR exists
for.

In [16]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 20, "lambda_mult": 0.5},
)

for doc in mmr_retriever.invoke("Medicare enrolment periods and deadlines"):
    print("- page", doc.metadata.get("page"), ":", doc.page_content[:110].replace("\n", " "))

`fetch_k` is how many candidates to consider, `lambda_mult` is the trade-off:
1.0 = pure relevance, 0.0 = maximum diversity.

#### 4.3. Score threshold

Returns nothing when nothing is good enough. This is your first line of defence against
hallucination — an empty result lets you answer "I don't know" instead of inventing.

In this domain that matters more than usual. A confidently invented coverage rule can send a
member to a service they will be billed for in full.

In [17]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 5},
)

for question in [
    "What is the Part B premium?",
    "What is the capital of Mongolia?",
    "What dose of metformin should I take?",
]:
    hits = threshold_retriever.invoke(question)
    print(f"\nQ: {question}")
    print(f"   {len(hits)} chunk(s) passed the threshold")
    for h in hits[:1]:
        print("   ->", h.page_content[:120].replace("\n", " "))

> Look at the third question. A clinical question may still retrieve chunks — the handbook talks
> about prescription drugs, so the vectors are not far apart. **Retrieval relevance is not the
> same as scope.** The threshold protects you from answering with no evidence; it does not
> protect you from answering a question you should refuse. That needs a separate check, which is
> what section 6.4 and notebook 4's router are for.

---
## 5. Generation — the RAG chain

The fastest RAG formulation: retrieve once, generate once. One LLM call per query, so it is
cheap and predictable. Use it when every question needs exactly one search.

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a health plan benefits assistant. Answer the question using ONLY the context "
     "below, which is drawn from the official Medicare & You handbook. "
     "If the context does not contain the answer, say you do not know and suggest the member "
     "call 1-800-MEDICARE. "
     "Never give clinical advice - if the question is about symptoms, diagnosis, treatment or "
     "medication dosing, decline and suggest the member speak to their doctor. "
     "Cite the page number for every fact you use. "
     "End by noting that specific plans may differ.\n\nContext:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    return "\n\n".join(
        f"[page {d.metadata.get('page')}] {d.page_content}" for d in docs
    )


rag_chain = (
    {"context": similarity_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

print(rag_chain.invoke("What is the late enrolment penalty for Part B and how long does it last?"))

#### 5.1. Test the guardrails

Two different failure modes, two different tests. The first question is in scope but not in the
corpus. The second is a clinical question that should be refused regardless of what retrieval
returns.

In [19]:
print("--- OUT OF CORPUS ---")
print(rag_chain.invoke("What is Medicare's policy on interplanetary travel benefits?"))
print()
print("--- OUT OF SCOPE (clinical) ---")
print(rag_chain.invoke("I have a rash on my arm. Which cream should I use for it?"))

---
## 6. The RAG agent

A chain always searches exactly once with the user's raw wording. An **agent** can rewrite the
query, search several times, and decide when it has enough. That costs more calls but handles
multi-part questions properly — and member questions are almost always multi-part.

We expose retrieval as a **tool** using `response_format="content_and_artifact"`, which lets
the tool return a readable string to the model *and* the original `Document` objects to your
application.

#### 6.1. Retrieval as a tool

In [20]:
from langchain_core.tools import tool


@tool(response_format="content_and_artifact")
def search_handbook(query: str):
    """Search the official Medicare & You handbook for benefits and coverage information.

    Use this for any question about Medicare coverage, costs, enrolment periods, premiums,
    deductibles, appeals, or the differences between Parts A, B, C and D.

    Rephrase the member's question into a focused search query using handbook terminology
    before calling. For example, turn "the yearly amount I pay first" into "annual deductible".
    """
    retrieved_docs = vector_store.similarity_search(query, k=3)
    serialised = "\n\n".join(
        f"Source: page {doc.metadata.get('page')}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialised, retrieved_docs


print(search_handbook.name)
print(search_handbook.description)

#### 6.2. Build the agent

In [21]:
from langchain.agents import create_agent

rag_agent = create_agent(
    model,
    tools=[search_handbook],
    system_prompt=(
        "You are a health plan benefits assistant with access to the official Medicare & You "
        "handbook. Always use the search tool before answering a benefits question - never "
        "answer from memory. You may search multiple times if the question has several parts. "
        "Cite page numbers for every fact. If the handbook does not cover something, say so and "
        "suggest calling 1-800-MEDICARE. "
        "Never give clinical advice: decline questions about symptoms, diagnosis, treatment "
        "choice or medication dosing and suggest the member speak to their doctor. "
        "Always note that a member's specific plan may differ from the general rules."
    ),
)

result = rag_agent.invoke({"messages": [
    {"role": "user", "content": "Does Medicare cover routine dental care?"}
]})

print(result["messages"][-1].content)

#### 6.3. Watch the agent search more than once

This question needs two lookups, and the second depends on the answer to the first. Stream the
run to see the agent decide that for itself.

In [22]:
query = (
    "What are the main enrolment periods for Medicare? "
    "Once you know them, look up what happens if I miss the first one."
)

for event in rag_agent.stream({"messages": [{"role": "user", "content": query}]},
                              stream_mode="values"):
    event["messages"][-1].pretty_print()

#### 6.4. The scope guardrail under pressure

A single clinical question is easy to refuse. The harder case is a **mixed** message: a real
benefits question wrapped around a clinical one. The assistant should answer the first part and
decline the second, rather than refusing everything or answering everything.

This is the test to keep in your evaluation set.

In [23]:
mixed = rag_agent.invoke({"messages": [
    {"role": "user", "content":
     "My doctor wants me to have an MRI. Is that covered under Part B, and do you think I "
     "actually need one for lower back pain?"}
]})

print(mixed["messages"][-1].content)

---
## 7. Returning source documents

Members and reviewers trust an answer far more when they can open the source. The agent's final
message is just text, so we add **middleware** that captures the retrieved documents into the
agent's state before the model runs.

For a compliance reviewer, this list *is* the evidence trail: it shows exactly which passages
the answer was generated from.

In [24]:
from typing import Any
from langchain_core.documents import Document
from langchain.agents.middleware import AgentMiddleware, AgentState


class State(AgentState):
    context: list[Document]


class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State

    def before_model(self, state: State) -> dict[str, Any] | None:
        last_message = state["messages"][-1]
        retrieved_docs = vector_store.similarity_search(last_message.text(), k=3)

        docs_content = "\n\n".join(
            f"[page {d.metadata.get('page')}] {d.page_content}" for d in retrieved_docs
        )
        augmented = (
            f"{last_message.text()}\n\n"
            "Use the following context to answer the query:\n"
            f"{docs_content}"
        )
        return {
            "messages": [last_message.model_copy(update={"content": augmented})],
            "context": retrieved_docs,
        }


sourced_agent = create_agent(
    model,
    tools=[],
    system_prompt=(
        "Answer using the provided context and cite page numbers. "
        "Do not give clinical advice. Note that specific plans may differ."
    ),
    middleware=[RetrieveDocumentsMiddleware()],
)
print("Agent with source tracking created")

In [25]:
response = sourced_agent.invoke(
    {"messages": [{"role": "user", "content": "What costs am I responsible for under Part B?"}]}
)

print("ANSWER:\n", response["messages"][-1].content)

print("\n--- SOURCE DOCUMENTS (the evidence trail) ---")
for i, doc in enumerate(response["context"], 1):
    print(f"\nDoc {i} (pdf page index {doc.metadata.get('page')}):")
    print(doc.page_content[:250].replace("\n", " "))

---
## 8. Tuning checklist

When RAG answers badly, the fix is almost never a better prompt. Work down this list:

| Symptom | Likely cause | Fix |
|---|---|---|
| Answer misses obvious facts | Chunks too small or too large | Tune `chunk_size` / `chunk_overlap` |
| Same paragraph returned repeatedly | Redundant corpus | Switch to `mmr` |
| Confident nonsense | Irrelevant chunks retrieved | Add `score_threshold`, instruct "say I don't know" |
| Right chunk exists but is not retrieved | Member wording differs from handbook wording | Use the agent so it can rewrite the query |
| Slow and expensive | `k` too high | Lower `k`, retrieve less, not more |
| Answers are stale | Index not rebuilt | Re-index on document change |
| Citation points at the wrong page | Loader page index vs printed page number | Store the printed number during ingestion |

#### 8.1. The staleness problem is not hypothetical here

The Medicare & You handbook is republished every year, and the premium and deductible figures
change with it. An index built last year will answer this year's questions with last year's
numbers — **confidently, and with a citation**, which makes it worse than an obvious error.

Anything you build on top of a document that has an annual cycle needs a re-indexing plan
before it goes anywhere near a member. At minimum:

- Store the document edition and ingestion date in the chunk metadata
- Surface the edition in the answer, not just the page number
- Fail loudly when the index is older than the current publication

---
## Summary

- Load → split → embed → store is the indexing half of RAG
- **Titan Text Embeddings V2** on Bedrock produces the vectors; **Chroma** stores them
- Embeddings bridge member language and document language — that is why this works at all
- Three retriever modes: similarity, MMR, score threshold
- A **RAG chain** is one search and one generation — fast and predictable
- A **RAG agent** searches as many times as the question needs
- Middleware puts source documents into state, giving you an evidence trail
- Retrieval relevance is not the same as scope: guardrails need their own check
- Documents with an annual cycle need a re-indexing plan

**Next:** notebook `4-LangGraph-State-Workflows.ipynb`, where we stop using the pre-built agent
and construct the graph ourselves — including an explicit scope router.